In [1]:
import torch


config = {
    # Модель
    "MODEL_NAME": "cisco-ai/SecureBERT2.0-base",  # или "jackaduma/SecureBERT" если первая недоступна

    # Пути
    "DATA_PATH": "data/output_ML_all.csv.gz",
    "SAVE_PATH": "cvss_model.pth",

    # Параметры обучения
    "BATCH_SIZE": 16,
    "EPOCHS": 5,
    "LEARNING_RATE": 2e-5,
    "MAX_LEN": 512,
    "WARMUP_STEPS": 100,
    "DEVICE": torch.device("cuda" if torch.cuda.is_available() else "cpu"),

    # TF-IDF параметры
    "TFIDF_MAX_FEATURES": 512,
    "TFIDF_NGRAM_RANGE": (1, 3),

    # Размерности
    "BERT_HIDDEN_SIZE": 768,  # Размер для SecureBERT/RoBERTa-base
    "TFIDF_FUSION_SIZE": 256,  # Размер после fusion слоя
}

# ==================== ОПРЕДЕЛЕНИЕ МЕТРИК CVSS ====================
CVSS_METRICS = {
    'attack_vector': {
        'classes': ['NETWORK', 'ADJACENT_NETWORK', 'LOCAL', 'PHYSICAL'],
        'num_classes': 4,
        'weight': 1.0
    },
    'attack_complexity': {
        'classes': ['LOW', 'HIGH'],
        'num_classes': 2,
        'weight': 1.0
    },
    'privileges_required': {
        'classes': ['NONE', 'LOW', 'HIGH'],
        'num_classes': 3,
        'weight': 1.5  # Больший вес из-за дисбаланса
    },
    'user_interaction': {
        'classes': ['NONE', 'REQUIRED'],
        'num_classes': 2,
        'weight': 1.0
    },
    'scope': {
        'classes': ['UNCHANGED', 'CHANGED'],
        'num_classes': 2,
        'weight': 1.0
    },
    'confidentiality': {
        'classes': ['NONE', 'LOW', 'HIGH'],
        'num_classes': 3,
        'weight': 1.2
    },
    'integrity': {
        'classes': ['NONE', 'LOW', 'HIGH'],
        'num_classes': 3,
        'weight': 1.2
    },
    'availability': {
        'classes': ['NONE', 'LOW', 'HIGH'],
        'num_classes': 3,
        'weight': 1.2
    }
}


In [2]:
import torch
import torch.nn as nn
from transformers import AutoModel


class SecureBERTWithTFIDF(nn.Module):
    """
    Multi-Task Learning модель для предсказания 8 метрик CVSS
    с fusion слоем для объединения эмбеддингов BERT и TF-IDF
    """

    def __init__(self, model_name, tfidf_dim, num_classes_per_metric):
        super().__init__()

        # SecureBERT encoder
        self.bert = AutoModel.from_pretrained(model_name)
        self.bert_hidden_size = self.bert.config.hidden_size

        # TF-IDF проекция (сжимаем до меньшей размерности)
        self.tfidf_projection = nn.Linear(tfidf_dim, config['TFIDF_FUSION_SIZE'])
        self.tfidf_dropout = nn.Dropout(0.3)

        # Fusion слой для объединения BERT и TF-IDF
        fusion_input_size = self.bert_hidden_size + config['TFIDF_FUSION_SIZE']
        self.fusion_layer = nn.Sequential(
            nn.Linear(fusion_input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Классификационные головки для каждой метрики (Multi-Task)
        self.classifiers = nn.ModuleDict()
        for metric_name, cvss_config in num_classes_per_metric.items():
            self.classifiers[metric_name] = nn.Sequential(
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(128, cvss_config['num_classes'])
            )

    def forward(self, input_ids, attention_mask, tfidf_features):
        # Получаем эмбеддинги из SecureBERT
        bert_outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Используем [CLS] токен (первый токен) как представление всего текста
        cls_embeddings = bert_outputs.last_hidden_state[:, 0, :]  # [batch_size, 768]

        # Проецируем TF-IDF признаки
        tfidf_projected = self.tfidf_projection(tfidf_features)  # [batch_size, 256]
        tfidf_projected = self.tfidf_dropout(tfidf_projected)

        # Конкатенируем BERT и TF-IDF представления
        combined = torch.cat([cls_embeddings, tfidf_projected], dim=1)  # [batch_size, 1024]

        # Fusion слой
        fused_features = self.fusion_layer(combined)  # [batch_size, 256]

        # Предсказания для каждой метрики
        outputs = {}
        for metric_name, classifier in self.classifiers.items():
            outputs[metric_name] = classifier(fused_features)

        return outputs



/home/zab/Git/desc2cvss/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.utils.data import Dataset


class CVEDataset(Dataset):
    def __init__(self, data, tokenizer, tfidf_vectorizer=None, max_len=512, fit_tfidf=False):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.tfidf_vectorizer = tfidf_vectorizer

        texts = self.data['description']

        if fit_tfidf:
            self.tfidf_vectorizer = TfidfVectorizer(
                max_features=config['TFIDF_MAX_FEATURES'],
                ngram_range=config['TFIDF_NGRAM_RANGE'],
                stop_words='english'
            )
            self.tfidf_features = self.tfidf_vectorizer.fit_transform(texts).toarray()
        else:
            self.tfidf_features = self.tfidf_vectorizer.transform(texts).toarray()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data.iloc[idx]
        description = item['description']

        # Токенизация
        encoding = self.tokenizer(
            description,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        # TF-IDF признаки
        tfidf = torch.tensor(self.tfidf_features[idx], dtype=torch.float32)

        # Метки для всех метрик
        labels = {}
        for metric_name in CVSS_METRICS.keys():
            labels[metric_name] = torch.tensor(
                item[metric_name],
                dtype=torch.long
            )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'tfidf_features': tfidf,
            'labels': labels
        }


In [4]:
class MultiTaskLoss(nn.Module):
    """
    Multi-Task Loss с автоматической настройкой весов задач
    на основе неопределенности (Kendall et al., 2018)
    """

    def __init__(self, num_tasks):
        super().__init__()
        # Обучаемые параметры log(sigma^2) для каждой задачи
        self.log_vars = nn.Parameter(torch.zeros(num_tasks))

    def forward(self, losses):
        total_loss = 0
        for i, loss in enumerate(losses):
            precision = torch.exp(-self.log_vars[i])
            total_loss += precision * loss + self.log_vars[i] / 2
        return total_loss


In [5]:
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import accuracy_score

from sklearn.metrics import f1_score


def compute_metrics(predictions, labels, metric_name):
    """Вычисление accuracy и F1 для конкретной метрики"""
    preds = torch.argmax(predictions, dim=1).cpu().numpy()
    true = labels.cpu().numpy()

    # Определяем тип усреднения для F1
    if CVSS_METRICS[metric_name]['num_classes'] > 2:
        average = 'macro'
    else:
        average = 'binary'

    return {
        'accuracy': accuracy_score(true, preds),
        'f1': f1_score(true, preds, average=average)
    }


def train_epoch(model, dataloader, optimizer, scheduler, criterion, device):
    """Одна эпоха обучения"""
    model.train()
    total_loss = 0
    all_predictions = {metric: [] for metric in CVSS_METRICS.keys()}
    all_labels = {metric: [] for metric in CVSS_METRICS.keys()}

    for batch in tqdm(dataloader, desc="Training"):
        # Перемещаем на устройство
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        tfidf_features = batch['tfidf_features'].to(device)
        labels = {k: v.to(device) for k, v in batch['labels'].items()}

        # Forward pass
        outputs = model(input_ids, attention_mask, tfidf_features)

        # Вычисляем потери для каждой задачи
        task_losses = []
        for metric_name in CVSS_METRICS.keys():
            loss = F.cross_entropy(outputs[metric_name], labels[metric_name])
            task_losses.append(loss)

            # Сохраняем для метрик
            all_predictions[metric_name].append(outputs[metric_name].detach())
            all_labels[metric_name].append(labels[metric_name])

        # Multi-task loss с автоматической настройкой весов
        loss = criterion(task_losses)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate(model, dataloader, device):
    """Оценка модели"""
    model.eval()
    metrics = {metric: {'accuracy': 0, 'f1': 0} for metric in CVSS_METRICS.keys()}

    all_predictions = {metric: [] for metric in CVSS_METRICS.keys()}
    all_labels = {metric: [] for metric in CVSS_METRICS.keys()}

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            tfidf_features = batch['tfidf_features'].to(device)
            labels = batch['labels']

            outputs = model(input_ids, attention_mask, tfidf_features)

            for metric_name in CVSS_METRICS.keys():
                all_predictions[metric_name].append(outputs[metric_name].cpu())
                all_labels[metric_name].append(labels[metric_name])

    # Вычисляем метрики для каждой задачи
    for metric_name in CVSS_METRICS.keys():
        preds = torch.cat(all_predictions[metric_name], dim=0)
        lbls = torch.cat(all_labels[metric_name], dim=0)

        task_metrics = compute_metrics(preds, lbls, metric_name)
        metrics[metric_name] = task_metrics

    return metrics

In [6]:
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

print(f"Using device: {config['DEVICE']}")

# Инициализация токенизатора
tokenizer = AutoTokenizer.from_pretrained(config['MODEL_NAME'])

# Создание TF-IDF векторайзера на тренировочных данных
print("Fitting TF-IDF vectorizer...")
data = pandas.read_csv(config['DATA_PATH'])

train_data, test_data = train_test_split(data, test_size=0.8, random_state=42, stratify=data['year'])
train_data, val_data = train_test_split(train_data, test_size=0.1, random_state=42, stratify=train_data['year'])

train_dataset = CVEDataset(train_data, tokenizer, fit_tfidf=True)
tfidf_vectorizer = train_dataset.tfidf_vectorizer


# Создание валидационного датасета
val_dataset = CVEDataset(val_data, tokenizer, tfidf_vectorizer=tfidf_vectorizer, fit_tfidf=False)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config['BATCH_SIZE'],
    shuffle=True,
    num_workers=4
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config['BATCH_SIZE'],
    shuffle=False,
    num_workers=4
)

# Инициализация модели
model = SecureBERTWithTFIDF(
    model_name=config['MODEL_NAME'],
    tfidf_dim=config['TFIDF_MAX_FEATURES'],
    num_classes_per_metric=CVSS_METRICS
).to(config['DEVICE'])

# Оптимизатор и scheduler
optimizer = AdamW(model.parameters(), lr=config['LEARNING_RATE'])
total_steps = len(train_loader) * config['EPOCHS']
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=config['WARMUP_STEPS'],
    num_training_steps=total_steps
)

# Multi-task loss с автоматической настройкой весов
criterion = MultiTaskLoss(num_tasks=len(CVSS_METRICS)).to(config['DEVICE'])

# Обучение
best_f1 = 0
for epoch in range(config['EPOCHS']):
    print(f"\n{'=' * 50}")
    print(f"Epoch {epoch + 1}/{config['EPOCHS']}")
    print(f"{'=' * 50}")

    # Training
    train_loss = train_epoch(
        model, train_loader, optimizer, scheduler, criterion, config['DEVICE']
    )
    print(f"Training Loss: {train_loss:.4f}")

    # Validation
    metrics = evaluate(model, val_loader, config['DEVICE'])

    # Вывод результатов
    print("\nValidation Metrics:")
    avg_f1 = 0
    for metric_name, metric_values in metrics.items():
        print(f"  {metric_name}: Acc={metric_values['accuracy']:.4f}, F1={metric_values['f1']:.4f}")
        avg_f1 += metric_values['f1']

    avg_f1 /= len(metrics)
    print(f"  Average F1: {avg_f1:.4f}")

    # Сохранение лучшей модели
    if avg_f1 > best_f1:
        best_f1 = avg_f1
        torch.save({
            'model_state_dict': model.state_dict(),
            'criterion_state_dict': criterion.state_dict(),
            'tfidf_vectorizer': tfidf_vectorizer,
            'metrics': metrics,
            'config': config
        }, config['SAVE_PATH'])
        print(f"✓ Model saved to {config['SAVE_PATH']}")

print(f"\nTraining complete! Best average F1: {best_f1:.4f}")


Using device: cuda
Fitting TF-IDF vectorizer...


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 12138.24it/s]
ModernBertModel LOAD REPORT from: cisco-ai/SecureBERT2.0-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Epoch 1/5


Training: 100%|██████████| 2761/2761 [35:33<00:00,  1.29it/s]


Training Loss: 3.9025


Evaluating: 100%|██████████| 307/307 [01:19<00:00,  3.87it/s]



Validation Metrics:
  attack_vector: Acc=0.8934, F1=0.4412
  attack_complexity: Acc=0.9085, F1=0.0000
  privileges_required: Acc=0.7738, F1=0.5298
  user_interaction: Acc=0.9122, F1=0.8586
  scope: Acc=0.9211, F1=0.7805
  confidentiality: Acc=0.8291, F1=0.8148
  integrity: Acc=0.8384, F1=0.8358
  availability: Acc=0.8533, F1=0.7982
  Average F1: 0.6324
✓ Model saved to cvss_model.pth

Epoch 2/5


Training:  12%|█▏        | 333/2761 [04:15<31:02,  1.30it/s]


KeyboardInterrupt: 